# MODFLOW 6 API D: a reverse drain

This is notebook D of the API series. Notebook A ([`mf6-api-a-basic`](mf6-api-a-basic.ipynb)) covers the steps of an API run and the outer-iteration loop, and notebook C ([`mf6-api-c-recharge-callback`](mf6-api-c-recharge-callback.ipynb)) covers callbacks.

A Drain (DRN) removes water from a cell when the head is above the drain elevation. A **reverse drain** does the opposite: it adds water when the head is below the elevation, which raises the head toward it,

$$q = C\,(E - h) \quad \text{when } h < E, \qquad q = 0 \text{ otherwise,}$$

where $q$ is the flow into the cell (L³/T), $C$ is the conductance (L²/T), $E$ is the elevation (L), and $h$ is the head in the cell (L).

MODFLOW 6 has no reverse drain, and a DRN package cannot be turned into one, because it recomputes its terms every outer iteration. The **API package** can. MODFLOW adds a head-dependent boundary to the equation of a cell as two terms: `HCOF`, which multiplies the head of the cell, and `RHS`, a constant. For a reverse drain that is adding water, `HCOF` is $-C$ and `RHS` is $-C E$. The API package leaves both to Python, so a callback can write them every outer iteration.

Here the elevations of the reverse drain are the water table of a one-dimensional unconfined aquifer with uniform recharge, worked out with the analytical solution. To hold the water table at that level, the reverse drain has to add water at the recharge rate that produced it. The model has no Recharge package, so the recharge the reverse drain adds can be checked against the rate the analytical water table was made with.

## What this notebook covers

By the end of this notebook you will be able to:

- add a head-dependent boundary of your own with the API package,
- make that boundary add water as well as remove it, which DRN cannot do, and
- compare the simulated water table with the analytical solution it should reproduce.

## Imports

Import FloPy, NumPy, and Matplotlib, and `Callbacks` and `run_simulation` from `modflowapi`. `mf6_api_helpers` finds the MODFLOW 6 library.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
from mf6_api_helpers import find_mf6_libraries
from modflowapi import Callbacks, run_simulation

## Find the MODFLOW 6 library

The API runs MODFLOW 6 through its shared library, `libmf6` (`libmf6.so` on Linux, `libmf6.dylib` on macOS, and `libmf6.dll` on Windows), rather than through the `mf6` program. pixi installs both with the class MODFLOW 6. `find_mf6_libraries()` returns the path of each.

In [ ]:
lib_path, mf6_exe = find_mf6_libraries()
print(f"library: {lib_path.name}")
print(f"program: {mf6_exe.name}")

## The analytical water table

Take a one-dimensional unconfined aquifer between a groundwater divide, where no water crosses, and a river, where the head is fixed. Recharge $N$ falls uniformly on it, so the water table mounds up between the two. Where the flow is horizontal (the Dupuit assumption), the steady water table solves

$$\frac{d}{dx}\left(K\,b\,\frac{db}{dx}\right) = -N,$$

where $K$ is the hydraulic conductivity (L/T), $b = h - z_\text{bot}$ is the saturated thickness (L), $z_\text{bot}$ is the bottom of the aquifer (L), and $x$ is the distance from the divide (L). With no flow at the divide, $x = 0$, and a saturated thickness of $b_L$ at the river, $x = L$, the solution is

$$b(x)^2 = b_L^2 + \frac{N}{K}\left(L^2 - x^2\right).$$

The model is a cross section one layer and one row deep, with 200 columns 5 m wide. The divide is the western edge, and the river is a Constant-Head (CHD) cell at a head of 0 m in the last column.

In [ ]:
sim_name = "reverse-drain"
model_name = "rdrn"
workspace = pl.Path("models/mf6-api-d-reverse-drain")

# 1-D cross section along the columns: 1 layer, 1 row, ncol columns (fine grid)
nlay, nrow, ncol = 1, 1, 200
delr = 5.0  # column width (m) -> 1000 m long cross section
delc = 10.0  # row width (m)
top = 10.0  # land surface (m)
botm = -2.0  # aquifer base (m); below the river head so the river cell stays wet

k = 1.0  # hydraulic conductivity (m/d)
recharge = 1.0e-4  # areal recharge the water table is in equilibrium with (m/d)
river_head = 0.0  # constant head at the river (last column)

cond = (
    1.0e3  # reverse-drain conductance, m²/d; large, so the heads stay at the elevations
)
# each outer iteration the reverse-drain terms move half way to their new values
damping = 0.5

# cell-center x, measured from the divide at the left edge (x = 0)
xcenters = (np.arange(ncol) + 0.5) * delr
x_river = xcenters[-1]  # the river (constant head) sits at the last cell

# steady 1-D Boussinesq (Dupuit) water table:
#   b(x)^2 = b_river^2 + (N/K)(x_river^2 - x^2),  b = head - botm
b_river = river_head - botm
analytical_head = botm + np.sqrt(
    b_river**2 + (recharge / k) * (x_river**2 - xcenters**2)
)

# the reverse drain is applied to every cell except the river cell; its elevation
# is the analytical water table there. With all cells active the reduced node
# numbers (1-based) equal the natural node order.
active_cols = np.arange(ncol - 1)
active_nodes = active_cols + 1
elevations = analytical_head[active_cols]

print(f"cross section length : {ncol * delr:.0f} m in {ncol} columns")
print(f"max analytical head  : {analytical_head.max():.3f} m at the divide")
print(f"river head           : {river_head:.3f} m at x = {x_river:.1f} m")

## Build the model

One steady-state groundwater flow model: no flow across the western edge, the default, a CHD river in the last column, and no Recharge package. The water the aquifer receives comes only from the reverse drain, an API package whose `HCOF` and `RHS` the callback writes while the model runs. The model starts from the analytical water table.

In [ ]:
def build_model(ws, averaging=None):
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name,
        sim_ws=ws,
        exe_name=mf6_exe,
        write_headers=False,
    )
    flopy.mf6.ModflowTdis(
        sim,
        nper=1,
        perioddata=[(1.0, 1, 1.0)],
        time_units="days",
    )
    flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        outer_maximum=2000,
        outer_dvclose=1.0e-8,
        inner_maximum=100,
        inner_dvclose=1.0e-10,
        linear_acceleration="BICGSTAB",
    )
    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=model_name,
        save_flows=True,
    )
    flopy.mf6.ModflowGwfdis(
        gwf,
        length_units="meters",
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=[botm],
    )
    # unconfined (convertible) so transmissivity follows the water table
    flopy.mf6.ModflowGwfnpf(
        gwf,
        icelltype=1,
        k=k,
        save_specific_discharge=True,
        # how saturated thickness and K are averaged between cells; None is the
        # default harmonic mean
        alternative_cell_averaging=averaging,
    )
    flopy.mf6.ModflowGwfic(
        gwf,
        strt=analytical_head.reshape(nlay, nrow, ncol),
    )
    # river: constant head at the last column
    flopy.mf6.ModflowGwfchd(
        gwf,
        stress_period_data=[[(0, 0, ncol - 1), river_head]],
        pname="river",
    )
    # the reverse drain is supplied through the API package: the callback writes
    # its HCOF and RHS each outer iteration and MF6 does not overwrite them
    flopy.mf6.ModflowGwfapi(
        gwf,
        pname="rdrn",
        maxbound=active_nodes.size,
        print_flows=True,
    )
    flopy.mf6.ModflowGwfoc(
        gwf,
        head_filerecord=f"{model_name}.hds",
        budget_filerecord=f"{model_name}.cbc",
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )
    return sim


sim = build_model(workspace)
sim.write_simulation(silent=True)
print("wrote simulation to", workspace)

## Run the reverse drain through the API

The callback does two things. Once, at `Callbacks.initialize`, it gives the API package its cells, by writing their node numbers (counted from 1) to its `NODELIST` and their number to `NBOUND`. Then, at the start of every outer iteration, it reads the current heads, `X`, and for every cell whose head is below its elevation sets `HCOF` to $-C$ and `RHS` to $-CE$, and both to zero elsewhere. Whether a cell is adding water can switch from one outer iteration to the next, so the callback moves each term only half way, `damping`, toward its new value, which keeps the heads from oscillating.

In [ ]:
MNAME = model_name.upper()


def reverse_drain(sim, step):
    """Supply the reverse-drain HCOF/RHS to the API package each outer iteration."""
    mf6 = sim.mf6
    if step == Callbacks.initialize:
        # assign the reverse drains to the active cells (constant for the run)
        nbound = mf6.get_value_ptr(mf6.get_var_address("NBOUND", MNAME, "RDRN"))
        nodelist = mf6.get_value_ptr(mf6.get_var_address("NODELIST", MNAME, "RDRN"))
        nbound[0] = active_nodes.size
        nodelist[:] = active_nodes

    if step == Callbacks.iteration_start:
        head = mf6.get_value_ptr(mf6.get_var_address("X", MNAME))
        hcof = mf6.get_value_ptr(mf6.get_var_address("HCOF", MNAME, "RDRN"))
        rhs = mf6.get_value_ptr(mf6.get_var_address("RHS", MNAME, "RDRN"))
        h = head[active_nodes - 1]
        injecting = h < elevations
        new_hcof = np.where(injecting, -cond, 0.0)
        new_rhs = np.where(injecting, -cond * elevations, 0.0)
        # under-relax (damp) the term before each solve for stability
        hcof[:] = (1.0 - damping) * hcof + damping * new_hcof
        rhs[:] = (1.0 - damping) * rhs + damping * new_rhs

In [ ]:
run_simulation(lib_path, workspace, reverse_drain, verbose=False)

## Compare with the analytical solution

Read the simulated heads from the head file and work out the water the reverse drain added to each cell from the final heads, $q = C\,(E - h)$. If the grid is fine enough:

1. the simulated water table lies on the analytical water table, and
2. the water the reverse drain added, divided by the cell area, equals the recharge $N$ the analytical water table was made with.

In [ ]:
def read_results(ws):
    """Return the simulated heads and the recharge the reverse drain added (m/d)."""
    h_sim = flopy.utils.HeadFile(ws / f"{model_name}.hds").get_data().flatten()
    q_drn = cond * np.maximum(elevations - h_sim[active_cols], 0.0)  # m³/d per cell
    return h_sim, q_drn / (delr * delc)


def summarize(h_sim, added):
    cell_area = delr * delc
    print(
        f"max |simulated - analytical| head : "
        f"{np.max(np.abs(h_sim[active_cols] - elevations)):.2e} m"
    )
    print(f"total reverse-drain inflow        : {added.sum() * cell_area:.4f} m³/d")
    print(
        f"total specified recharge (N*area) : "
        f"{recharge * cell_area * active_nodes.size:.4f} m³/d"
    )
    print(
        f"added recharge per cell           : {added.min() * 1e3:.4f} to "
        f"{added.max() * 1e3:.4f} mm/d"
    )


h_sim, added = read_results(workspace)
summarize(h_sim, added)

In [ ]:
def plot_reverse_drain(h_sim, added):
    """Plot the water table and the recharge the reverse drain added."""
    with flopy.plot.styles.USGSPlot():
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(8, 6.5), sharex=True, constrained_layout=True
        )

        # A. water table
        ax1.fill_between(xcenters, botm, top, color="0.93", zorder=0)
        ax1.plot(
            xcenters,
            analytical_head,
            color="0.2",
            lw=2.0,
            label="Analytical (Boussinesq)",
        )
        ax1.plot(
            xcenters[::8],
            h_sim[::8],
            ls="none",
            marker="o",
            ms=5,
            mfc="none",
            mec="#eb6834",
            mew=1.2,
            label="MODFLOW (reverse drain)",
        )
        ax1.axhline(botm, color="saddlebrown", lw=1.2, ls="--", label="Aquifer base")
        ax1.plot(
            [x_river],
            [river_head],
            ls="none",
            marker="v",
            ms=11,
            mfc="#2a78d6",
            mec="k",
            mew=0.6,
            clip_on=False,
            zorder=6,
            label="River (constant head)",
        )
        ax1.annotate(
            "divide\n(no flow)",
            xy=(0, analytical_head[0]),
            xytext=(60, analytical_head[0] - 2.0),
            fontsize=9,
            arrowprops=dict(arrowstyle="->", lw=0.8),
        )
        ax1.set_ylim(botm - 0.5, top)
        flopy.plot.styles.heading(ax=ax1, letter="A", heading="Water table")
        flopy.plot.styles.ylabel(ax=ax1, label="Head (m)")
        ax1.legend(loc="upper right", frameon=False, fontsize=9)

        # B. recharge added by the reverse drain
        ax2.plot(
            xcenters[active_cols],
            added * 1e3,
            color="#eb6834",
            lw=1.5,
            label="Reverse-drain inflow / area",
        )
        ax2.axhline(
            recharge * 1e3,
            color="0.2",
            lw=2.0,
            ls="--",
            label="Specified recharge $N$",
        )
        ax2.set_ylim(0, recharge * 1e3 * 1.6)
        ax2.set_xlim(0, 1000)
        flopy.plot.styles.heading(
            ax=ax2, letter="B", heading="Recharge reproduced by the reverse drain"
        )
        flopy.plot.styles.xlabel(ax=ax2, label="Distance from divide, x (m)")
        flopy.plot.styles.ylabel(ax=ax2, label="Recharge (mm/d)")
        ax2.legend(loc="lower center", frameon=False, fontsize=9)
        fig.align_ylabels((ax1, ax2))


plot_reverse_drain(h_sim, added)

**What to look for.** The simulated water table (A) lies on the analytical water table, within 0.00001 m in every cell, from 8.17 m at the divide to 0 m at the river. The water the reverse drain added per unit area (B) equals the recharge of 0.1 millimeters per day (mm/d) in every cell except the last few before the river, where it falls to 0.079 mm/d. The simulated heads match the analytical heads there, so the reverse drain is adding less water because less water flows from cell to cell toward the river than the analytical solution says should.

## Investigate: averaging saturated thickness between cells

MODFLOW works out the flow between two cells from a conductance that combines the hydraulic conductivity and saturated thickness of both cells. By default it uses the harmonic mean of the two transmissivities, $Kb$. Under the Dupuit assumption, the flow between two points 5 m apart is exactly

$$Q = K\,\frac{b_1 + b_2}{2}\,\frac{h_1 - h_2}{\Delta x}\,\Delta y,$$

where $b_1$ and $b_2$ are the saturated thicknesses at the two points (L), $h_1$ and $h_2$ are the heads (L), $\Delta x$ is the distance between them (L), and $\Delta y$ is the width of the cross section (L). That is the *arithmetic* mean thickness. The harmonic mean is always smaller, and the difference grows where the thickness changes fastest relative to its value. Here that is next to the river, where the saturated thickness falls to 2 m.

The `ALTERNATIVE_CELL_AVERAGING` option of the Node Property Flow (NPF) package chooses another mean:

- `LOGARITHMIC`: the logarithmic mean of the transmissivities,
- `AMT-LMK`: the arithmetic mean saturated thickness and the logarithmic mean hydraulic conductivity, and
- `AMT-HMK`: the arithmetic mean saturated thickness and the harmonic mean hydraulic conductivity.

Before running the next cell, decide which you expect to add the specified recharge in every cell, and whether `AMT-LMK` and `AMT-HMK` should differ in this aquifer. Each option runs in its own folder with the same callback.

In [ ]:
results = {"HARMONIC": (h_sim, added)}
for averaging in ("LOGARITHMIC", "AMT-LMK", "AMT-HMK"):
    ws = pl.Path(f"{workspace}-{averaging.lower()}")
    build_model(ws, averaging).write_simulation(silent=True)
    run_simulation(lib_path, ws, reverse_drain, verbose=False)
    results[averaging] = read_results(ws)

for averaging, (h, q) in results.items():
    print(averaging)
    summarize(h, q)
    print()

In [ ]:
colors = {
    "HARMONIC": "#eb6834",
    "LOGARITHMIC": "#8a5cc2",
    "AMT-LMK": "#3fa34d",
    "AMT-HMK": "#2a78d6",
}
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
    for averaging, (_, q) in results.items():
        ls = ":" if averaging == "AMT-HMK" else "-"
        ax.plot(
            xcenters[active_cols],
            q * 1e3,
            color=colors[averaging],
            lw=1.5,
            ls=ls,
            label=averaging,
        )
    ax.axhline(recharge * 1e3, color="0.2", lw=1.0, ls="--", label="Specified recharge")
    ax.set_xlim(800, 1000)
    ax.set_ylim(0.07, 0.105)
    flopy.plot.styles.heading(ax=ax, heading="Recharge added near the river")
    flopy.plot.styles.xlabel(ax=ax, label="Distance from divide, x (m)")
    flopy.plot.styles.ylabel(ax=ax, label="Recharge (mm/d)")
    ax.legend(loc="lower left", frameon=False, fontsize=9)

**What to look for.** Next to the river, the reverse drain adds 0.079 mm/d with the harmonic mean and 0.093 mm/d with the logarithmic mean. With either arithmetic mean thickness, `AMT-LMK` or `AMT-HMK`, it adds between 0.0997 and 0.1002 mm/d in every cell, the specified recharge to within 0.3 percent. The two give the same answer because the hydraulic conductivity is the same in every cell, so every mean of it is the same. The arithmetic mean thickness reproduces the Dupuit flow between cells exactly, so the heads and the recharge both match the analytical solution.

Some things to try:

- Raise the aquifer base, `botm`, to -0.5 m, so the river cell is thinner. Does the harmonic mean do better or worse?
- Use 50 columns 20 m wide instead of 200. Which errors grow?
- Make the hydraulic conductivity change along the cross section. Do `AMT-LMK` and `AMT-HMK` still agree?

In [ ]:
plot_reverse_drain(*results["AMT-HMK"])

**What to look for.** With `AMT-HMK`, the simulated water table (A) still lies on the analytical water table, and the recharge the reverse drain added (B) is the specified 0.1 mm/d all the way to the river.

## Recap

You added a boundary MODFLOW 6 does not have, a drain that adds water, by writing the `HCOF` and `RHS` terms of an API package from a callback every outer iteration and moving them only part way each time. Holding the water table on its analytical profile, the reverse drain added the recharge that produced the profile, which checks the boundary against a known answer. The check also showed that the default harmonic mean of transmissivity carries too little water toward the river where the saturated thickness changes quickly, and that an arithmetic mean saturated thickness, `AMT-HMK`, removes the error.